In [2]:
import pandas as pd

df = pd.read_json("../tanner_report.json", lines=True)

df.head(5)

,method,path,headers,uuid,peer,status,cookies,response_msg,timestamp,post_data
0,GET,/,"{'host': 'localhost', 'user-agent': 'curl/7.81...",50d7b1f4-6ef6-4338-b372-dbd50f852dd9,"{'ip': '172.18.0.1', 'port': 59950}",200,{'sess_uuid': None},"{'version': '0.6.0', 'response': {'message': {...",2025-06-08 10:53:33.297062,NaN
1,GET,/,"{'host': '127.0.0.1', 'user-agent': 'Mozilla/5...",50d7b1f4-6ef6-4338-b372-dbd50f852dd9,"{'ip': '172.18.0.1', 'port': 38278}",200,{'sess_uuid': None},"{'version': '0.6.0', 'response': {'message': {...",2025-06-08 10:53:45.782025,NaN
2,GET,/,"{'host': '127.0.0.1', 'user-agent': 'Mozilla/5...",50d7b1f4-6ef6-4338-b372-dbd50f852dd9,"{'ip': '172.18.0.1', 'port': 38278}",200,{'sess_uuid': None},"{'version': '0.6.0', 'response': {'message': {...",2025-06-08 10:53:45.872072,NaN
3,GET,/robots.txt,"{'host': '127.0.0.1', 'user-agent': 'Mozilla/5...",50d7b1f4-6ef6-4338-b372-dbd50f852dd9,"{'ip': '172.18.0.1', 'port': 38286}",200,{'sess_uuid': None},"{'version': '0.6.0', 'response': {'message': {...",2025-06-08 10:53:45.891330,NaN
4,GET,/sitemap.xml,"{'host': '127.0.0.1', 'user-agent': 'Mozilla/5...",50d7b1f4-6ef6-4338-b372-dbd50f852dd9,"{'ip': '172.18.0.1', 'port': 38288}",200,{'sess_uuid': None},"{'version': '0.6.0', 'response': {'message': {...",2025-06-08 10:53:45.893769,NaN


In [3]:
method = df["method"]

methods_matrix = []
tmp = []

for i, x in enumerate(method):
  if i % 10 == 0 and i != 0:
    methods_matrix.append(tmp)
    tmp = []
  tmp.append(x)

# Add the last group if not empty
if tmp:
  methods_matrix.append(tmp)

methods_matrix

[['GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET'],
 ['GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET'],
 ['GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET'],
 ['GET', 'POST', 'POST', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET'],
 ['GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET'],
 ['GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'POST', 'POST', 'GET', 'GET'],
 ['POST', 'POST', 'POST', 'GET', 'POST', 'GET', 'POST', 'GET', 'POST', 'GET'],
 ['GET', 'POST', 'GET', 'POST', 'GET', 'GET', 'POST', 'GET', 'POST', 'GET'],
 ['POST', 'GET', 'GET', 'GET', 'POST', 'GET', 'GET', 'POST', 'GET', 'GET'],
 ['POST', 'GET', 'GET', 'POST', 'GET', 'POST', 'GET', 'GET', 'GET', 'GET'],
 ['POST', 'GET', 'GET', 'GET', 'GET', 'GET', 'POST', 'GET', 'GET', 'GET'],
 ['GET', 'GET', 'GET', 'POST', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET'],
 ['GET', 'GET', 'POST', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET'],
 ['POST', 'GET', 'GET', 'GE

In [4]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpmax, fpgrowth

te = TransactionEncoder()
te_ary = te.fit(methods_matrix).transform(methods_matrix)
df = pd.DataFrame(te_ary, columns=te.columns_)

df

,GET,HEAD,OPTIONS,POST,PROPFIND,PUT,SEARCH,TRACE
0,True,False,False,False,False,False,False,False
1,True,False,False,False,False,False,False,False
2,True,False,False,False,False,False,False,False
3,True,False,False,True,False,False,False,False
4,True,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...
1141,True,False,False,False,False,True,False,False
1142,True,False,False,False,False,False,False,False
1143,True,False,False,False,False,False,False,False
1144,True,False,True,False,False,False,False,False


In [12]:
# frequent_itemsets = fpgrowth(df, min_support=0.3, use_colnames=True)
frequent_itemsets = fpgrowth(df, min_support=0.005, use_colnames=True)
### alternatively:
#frequent_itemsets = apriori(df, min_support=0.6, use_colnames=True)
#frequent_itemsets = fpmax(df, min_support=0.6, use_colnames=True)

frequent_itemsets.head(20)

,support,itemsets
0,0.996510,(GET)
1,0.286213,(POST)
2,0.009599,(HEAD)
3,0.009599,(PUT)
4,0.010471,(OPTIONS)
5,0.009599,(PROPFIND)
6,0.009599,(TRACE)
7,0.283595,"(GET, POST)"
8,0.009599,"(GET, HEAD)"
9,0.009599,"(GET, PUT)"


In [13]:
import psycopg2

conn = psycopg2.connect(database="web_honeypot_generated", user = "postgres", password = "admin", host = "127.0.0.1", port = "5432")

print("Opened database successfully")

Opened database successfully


In [14]:
# create table
cur = conn.cursor()
cur.execute('''CREATE TABLE assoc_rules_methods (
            ID INT PRIMARY KEY     NOT NULL,
            SUPPORT           REAL    NOT NULL,
            METHOD            VARCHAR(255)     NOT NULL);''')

print("Table created successfully")

conn.commit()

Table created successfully


In [15]:
# insert data

cur = conn.cursor()

insert_query = """
    INSERT INTO assoc_rules_methods (ID, SUPPORT, METHOD)
    VALUES (%s, %s, %s)
"""

for idx, row in frequent_itemsets.iterrows():
    cur.execute(
        insert_query,
        (int(idx), float(row["support"]), str(list(row["itemsets"])))
    )

conn.commit()
print("Records created successfully")
conn.close()

Records created successfully
